# EDA i Feature Engineering: ściąga

Jedna komórka = jedna technika. Używaj jako punktu odniesienia podczas pracy z nowym dataselem.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "order_id":  ["A","B","C","D","E","F","G","H","I","J"],
    "price":     [120, None, 80, 150, 40, None, 250, 90, 30, 200],
    "qty":       [  1,    2,  1,   1,  3,    1,   1,  2,  1,   1],
    "days_late": [  0,    2, None, 7,  1,    0,   3, None, 0,   5],
    "category":  ["shoes","clothes","shoes", None,"shoes",
                  "clothes","electronics","clothes","shoes","electronics"],
    "is_bad":    [0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
})

print(df.to_string())

## Braki danych: wykrywanie

In [ ]:
# Liczba braków per kolumna
print(df.isnull().sum())
print()

# Procent braków
print((df.isnull().sum() / len(df) * 100).round(1).astype(str) + " %")
print()

# Przegląd typów i braków jednocześnie
df.info()

## Braki danych: usuwanie

In [ ]:
# Usuń wiersze gdzie JAKIEKOLWIEK pole ma brak
df_dropped_any = df.dropna()
print(f"dropna():              {len(df)} → {len(df_dropped_any)} wierszy")

# Usuń wiersze tylko jeśli braki w konkretnych kolumnach
df_dropped_price = df.dropna(subset=["price"])
print(f"dropna(subset=price):  {len(df)} → {len(df_dropped_price)} wierszy")

# Usuń kolumny z >30% braków
threshold = 0.3
cols_to_drop = df.columns[df.isnull().mean() > threshold].tolist()
print(f"Kolumny >30% braków:   {cols_to_drop}")

## Braki danych: wypełnianie

In [ ]:
df2 = df.copy()

# Stała wartość
df2["category"] = df2["category"].fillna("unknown")

# Mediana (odporna na outliery: lepsza niż mean dla skośnych rozkładów)
df2["price"] = df2["price"].fillna(df2["price"].median())

# Średnia
df2["days_late"] = df2["days_late"].fillna(df2["days_late"].mean())

# Moda (najczęstsza wartość)
# df2["category"] = df2["category"].fillna(df2["category"].mode()[0])

# Forward fill: ostatnia znana wartość (szeregi czasowe)
# df2["price"] = df2["price"].ffill()

print(df2.isnull().sum())

## Podstawowe statystyki

In [ ]:
# Rozkład statystyczny kolumn numerycznych
print(df2[["price", "qty", "days_late"]].describe().round(2))

In [ ]:
# Rozkład kategorii: ile wystąpień
print(df2["category"].value_counts())
print()

# Ile unikalnych wartości per kolumna
print(df2.nunique())

## Histogram: rozkład zmiennej ciągłej

In [ ]:
import plotly.express as px

fig = px.histogram(df2, x="price", nbins=8, title="Rozkład cen zamówień",
                   labels={"price": "Cena (PLN)"})
fig.show()

In [ ]:
# Dwie kolumny obok siebie
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Cena", "Dni opóźnienia"])
fig.add_trace(go.Histogram(x=df2["price"],     name="price",     nbinsx=8), row=1, col=1)
fig.add_trace(go.Histogram(x=df2["days_late"], name="days_late", nbinsx=6), row=1, col=2)
fig.update_layout(showlegend=False, title="Rozkłady cech numerycznych")
fig.show()

## Korelacje

In [ ]:
corr = df2[["price", "qty", "days_late", "is_bad"]].corr().round(2)
print(corr)

In [ ]:
fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="Macierz korelacji")
fig.show()

## Outliery: boxplot i metoda IQR

**Boxplot = IQR narysowany.**

```
     |------|=====|=======|=====|------|   o   o
          Q1    median   Q3
     ^                         ^           ^
  dolny wąs                górny wąs    outliery
```

- **Q1** = 25. percentyl: 25% danych leży poniżej
- **Q3** = 75. percentyl: 75% danych leży poniżej
- **IQR** = Q3 − Q1 = szerokość środkowych 50% danych
- **Wąsy** = Q1 − 1.5·IQR  i  Q3 + 1.5·IQR
- Punkty **poza wąsami** = outliery

In [ ]:
# Boxplot: wizualizacja IQR
fig = px.box(df2, y="price", points="all",
             title="Boxplot cen, punkty poza wąsami to outliery",
             labels={"price": "Cena (PLN)"})
fig.show()

## Outliery: obliczenie granic IQR (to samo co boxplot)

In [ ]:
col = df2["price"]

Q1  = col.quantile(0.25)
Q3  = col.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1={Q1:.1f}  Q3={Q3:.1f}  IQR={IQR:.1f}")
print(f"Granice: [{lower:.1f}, {upper:.1f}]")
print()

outliers = df2[col > upper]
print(f"Outliery powyżej górnej granicy ({len(outliers)} wierszy):")
print(outliers[["order_id", "price"]])

## Co z outlierami zrobić?

Trzy opcje, wybór zależy od kontekstu:

| Opcja | Kiedy |
|---|---|
| **Usuń wiersz** | błąd w danych, np. cena = -50 |
| **Przytnij (clip)** | wartość realna, ale ekstremalna - nie chcemy żeby dominowała |
| **Zostaw** | model drzewiasty (XGBoost) - i tak sobie poradzi |

In [ ]:
# Przycinanie do granic IQR
df_clipped = df2.copy()
df_clipped["price"] = df_clipped["price"].clip(lower=lower, upper=upper)

print("Przed clip:", df2["price"].tolist())
print("Po clip:   ", df_clipped["price"].tolist())

## Feature Engineering: przykładowe techniki

In [ ]:
fe = df2.copy()

# 1. Cecha ratio
fe["price_per_item"] = fe["price"] / fe["qty"].clip(lower=1)

# 2. Cecha binarna z warunku
fe["is_late"] = (fe["days_late"] > 0).astype(int)

# 3. Binning: podział na kubełki
fe["price_bin"] = pd.cut(fe["price"],
                          bins=[0, 50, 100, 200, float("inf")],
                          labels=["tani", "sredni", "drogi", "premium"])

# 4. Log transform: spłaszczenie skośnego rozkładu
fe["log_price"] = np.log1p(fe["price"])

print(fe[["order_id","price","price_per_item","is_late","price_bin","log_price"]].to_string())

In [ ]:
# 5. One-Hot Encoding
dummies = pd.get_dummies(fe["category"], prefix="cat", dtype=int)
fe_ohe = pd.concat([fe, dummies], axis=1)
print(fe_ohe[["order_id","category"] + list(dummies.columns)].to_string())